# Theme 2: RAG Application, Starter Kit

This starter kit contains a complete, working **Retrieval-Augmented Generation (RAG)** application using **Gemma 4**, Google's open-weight model. Your team's task is to adapt and extend it for the problem statement you selected (2A or 2B).

**How RAG works:**

```
 Document ──► 1. split into chunks ──► 2. convert each chunk into an embedding (a list of numbers)
                                                           │
 Question ──► 3. retrieve the chunks whose embeddings are closest to the question's embedding
                                                           │
              4. give ONLY those chunks to Gemma ──► answer with page citations
```

**Models used (both open-weight):**
- `all-MiniLM-L6-v2` (sentence-transformers) creates the embeddings. It runs inside this notebook.
- `gemma-4-26b-a4b-it` (Google) writes the answers. It is accessed through the free Google AI Studio API.

**No GPU is required.** Use the default Colab runtime (CPU).

### Obtaining a free Google AI Studio API key (one per team)
1. Go to **https://aistudio.google.com/apikey** and sign in with a Google account.
2. Select **Create API key** and copy the key.
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GOOGLE_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it.
>
> **No key yet:** the notebook still runs. The retrieval stages work normally, and the application shows the passages it found in place of a written answer.

### How to use this notebook
- Run the cells **in order, from top to bottom**, the first time (select a cell and press **Shift + Enter**).
- Comments marked **STUDENT TODO** show the intended places to make changes.
- After changing a cell, re-run that cell and any cells below it that depend on it.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

## Step 0: Installation (run once)

In [52]:
# Installs the required libraries. This takes about one minute.
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

## Step 1: Settings

These values control how documents are divided, searched and answered. They are the main parameters to experiment with.

In [53]:
# STUDENT TODO: the Gemma 4 model that writes the answers.
#   "gemma-4-26b-a4b-it"  Mixture of Experts, 26B total / about 4B active parameters. Fast. (default)
#   "gemma-4-31b-it"      Dense, 31B parameters. More capable, slower.
GEMMA_MODEL = "gemma-4-26b-a4b-it"

# STUDENT TODO: the number of words in each chunk.
#   Small chunks (50)   more precise matches, but may lose context
#   Large chunks (300)  more context, but less precise and longer prompts
CHUNK_SIZE = 120

# STUDENT TODO: the number of words shared by neighbouring chunks, so sentences are not split.
CHUNK_OVERLAP = 30

# STUDENT TODO: the number of chunks given to Gemma for each question.
TOP_K = 3

# STUDENT TODO: the rules Gemma follows when answering. The rule to use only the context prevents invented answers.
ANSWER_RULES = """You are a document question-answering assistant.

Use ONLY the context passages provided to answer the question.
Each context passage begins with its source page label, such as [Page 2].

Rules:
- Answer using only information supported by the supplied context.
- Cite every factual claim with its source page, using the format (p. 2).
- If multiple pages support a claim, cite them like (pp. 2, 4).
- Never invent page numbers, facts, or sources.
- If the answer is not supported by the context, reply exactly:
  "The answer could not be found in the document."
- Keep answers concise, usually 2 to 4 sentences.
- Do not claim to have checked information outside the supplied context.
"""

TEMPERATURE = 0.2       # a low value keeps answers consistent and close to the document

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # STUDENT TODO (optional): try "BAAI/bge-small-en-v1.5"
print("Settings configured.")

Settings configured.


## Step 2: Connect to Gemma and load the embedding model

In [54]:
import os
from google import genai
from google.genai import types, errors


def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY")
DEMO_MODE = not GOOGLE_API_KEY
client = None if DEMO_MODE else genai.Client(api_key=GOOGLE_API_KEY)


def _friendly_error(e):
    code = getattr(e, "code", None)
    if code in (400, 401, 403) and "key" in str(e).lower():
        return "Error: the GOOGLE_API_KEY was rejected. Check that the complete key is saved in Colab Secrets, then re-run this cell."
    if code == 429:
        return "Error: the free-tier usage limit has been reached. Wait about one minute and try again."
    if code == 404:
        return f"Error: the model '{GEMMA_MODEL}' was not found. Check the GEMMA_MODEL setting."
    return f"Error: the request to Gemma failed ({e})."


def generate(user_text, system_text=None, temperature=TEMPERATURE, demo_reply="(Demonstration mode response)"):
    """Sends one message to Gemma and returns its reply as text.
    In demonstration mode, returns `demo_reply`. If the request fails, returns a message beginning with "Error:"."""
    if DEMO_MODE:
        return demo_reply
    try:
        response = client.models.generate_content(
            model = GEMMA_MODEL,
            contents = user_text,
            config = types.GenerateContentConfig(
                system_instruction = system_text,
                temperature = temperature,
                thinking_config = types.ThinkingConfig(thinking_level="minimal"),  # faster replies
            ),
        )
        return (response.text or "").strip()
    except errors.APIError as e:
        return _friendly_error(e)
    except Exception as e:
        return f"Error: {e}"


if DEMO_MODE:
    print("Demonstration mode: no GOOGLE_API_KEY was found. Retrieval will work; answers will show the passages found.")
else:
    print(f"Connected to Google AI Studio. Model: {GEMMA_MODEL}")
    print(generate("In one sentence, what is an open-weight AI model?"))

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME)   # runs on the CPU
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

Connected to Google AI Studio. Model: gemma-4-26b-a4b-it
An open-weight AI model is a machine learning model where the trained parameters (weights) are made publicly available, allowing developers to download, run, and fine-tune the model on their own hardware.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


## Step 3: Sample document
A short, fictional course syllabus is provided so the application works immediately. Each item in the list is one page. Your own PDF can be uploaded in the application in Step 5.

In [55]:
# STUDENT TODO: replace with text relevant to your problem statement, or upload a PDF in the application instead.
SAMPLE_PAGES = [
    """CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu.
    Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM
    in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces
    programming using Python. No prior coding experience is required.""",

    """Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. Final exam: 25%.
    Class participation: 5%. A score of 90% or above earns an A, 80 to 89% a B, 70 to 79% a C, and 60 to 69% a D.
    Late submissions lose 10% per day, up to a maximum of 3 days, after which they receive zero. Each student has
    two grace days per semester that may be used to submit any lab late without penalty.""",

    """Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and loops. Week 5: functions.
    Week 6: lists and dictionaries. Week 7: Midterm exam (in class, closed book). Weeks 8-9: files and error handling.
    Weeks 10-11: an introduction to APIs and working with data. Weeks 12-13: mini-project 2, a team project of up to
    3 students. Week 14: review. The final exam is a 2-hour written exam during the examination period.""",

    """Policies. Attendance: students must attend at least 75% of lectures to be eligible for the final exam.
    Academic integrity: students may discuss ideas with classmates, but all submitted code must be their own. AI
    tools may be used for learning and debugging, but their use must be stated in a comment at the top of the file.
    Laptops: bring a laptop to every lab. Students without a laptop may reserve a machine in the Computer Lab.
    Textbook: "Automate the Boring Stuff with Python" (available free online).""",
]
print(f"Sample document contains {len(SAMPLE_PAGES)} pages.")

Sample document contains 4 pages.


## Step 4: The RAG pipeline
Each function is one stage of the pipeline. This is the core of the project and the main code to extend.

In [56]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}  # populated by build_knowledge_base()


def read_pdf(path):
    """Returns a list of page texts from a PDF file."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]

import os

def read_document(path):
    """Read PDF, Markdown, or plain-text files."""
    extension = os.path.splitext(path)[1].lower()

    if extension == ".pdf":
        return read_pdf(path), True

    if extension in [".md", ".txt"]:
        with open(path, "r", encoding="utf-8") as file:
            text = file.read()
        if not text.strip():
            raise ValueError("The document is empty.")
        return [text], False

    raise ValueError("Supported formats: PDF, Markdown (.md), and text (.txt).")




def split_into_chunks(pages, name, is_pdf=True):
    """Split documents into chunks while preserving source information."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)

    for page_number, text in enumerate(pages, start=1):
        words = text.split()

        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]

            if len(chunk_words) > 5:
                chunks.append({
                    "source": name,
                    "page": page_number if is_pdf else None,
                    "text": " ".join(chunk_words)
                })

            if start + CHUNK_SIZE >= len(words):
                break

    return chunks


def build_knowledge_base(pages, name, is_pdf=True):
    """Create chunks and embeddings for a document."""
    chunks = split_into_chunks(pages, name, is_pdf)

    if not chunks:
        raise ValueError(
            "No readable text was found in the document."
        )

    vectors = embedder.encode(
        [c["text"] for c in chunks],
        normalize_embeddings=True
    )

    knowledge_base.update(
        name=name,
        chunks=chunks,
        vectors=vectors
    )

    return (
        f"Loaded **{name}**: {len(pages)} "
        f"{'pages' if is_pdf else 'text section(s)'}, "
        f"{len(chunks)} chunks."
    )


def answer(question):
    """Retrieve relevant chunks and ask Gemma to answer."""
    found = retrieve(question)

    context_parts = []

    for c in found:
        if c["page"] is not None:
            label = f"[Source: {c['source']}, Page {c['page']}]"
        else:
            label = f"[Source: {c['source']}]"

        context_parts.append(f"{label}\n{c['text']}")

    context = "\n\n".join(context_parts)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"

    demo = (
        "Demonstration mode: Gemma did not generate an answer "
        "because no API key is set. The most relevant source is "
        f"{found[0]['source']}."
    )

    reply = generate(
        prompt,
        system_text=ANSWER_RULES,
        demo_reply=demo
    )

    return reply, found



# Quick test with the sample document.
print(build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"))
test_question = "What happens if I submit a lab late?"
reply, found = answer(test_question)
print(f"\nQuestion: {test_question}\nAnswer: {reply}\n\nRetrieved passages:")
for c in found:
    print(f"   [page {c['page']}, similarity {c['score']:.2f}] {c['text'][:90]}...")

Loaded **Sample CS 101 syllabus**: 4 pages, 4 chunks.

Question: What happens if I submit a lab late?
Answer: Late submissions lose 10% per day for up to a maximum of 3 days, after which they receive a score of zero (p. 2). However, each student is provided two grace days per semester that can be used to submit any lab late without penalty (p. 2).

Retrieved passages:
   [page 2, similarity 0.53] Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. ...
   [page 4, similarity 0.33] Policies. Attendance: students must attend at least 75% of lectures to be eligible for the...
   [page 3, similarity 0.28] Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and lo...


## Step 5: Launch the application
Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.

In [58]:
import os
import gradio as gr

gr.close_all()

APP_TITLE = "OpenSource Navigator"
APP_DESCRIPTION = (
    "Ask questions about open-source project documentation. "
    "Get answers with source citations and retrieved passages."
)


def on_upload(file):
    if file is None:
        return "No file uploaded."

    try:
        pages, is_pdf = read_document(file)

        status = build_knowledge_base(
            pages,
            os.path.basename(file),
            is_pdf=is_pdf
        )

        return status

    except Exception as e:
        raise gr.Error(f"The document could not be read: {e}")


def on_ask(question):
    if not question or not question.strip():
        raise gr.Error("Enter a question first.")

    if knowledge_base["vectors"] is None:
        raise gr.Error("Upload a document or select the sample document first.")

    reply, found = answer(question)
    source_parts = []

    for c in found:
        if c["page"] is not None:
            label = f"**{c['source']} — Page {c['page']}**"
        else:
            label = f"**Source: {c['source']}**"

        source_parts.append(
            f"{label} (similarity {c['score']:.2f})\n\n"
            f"> {c['text']}"
        )

    return reply, "\n\n---\n\n".join(source_parts)


with gr.Blocks(title=APP_TITLE) as demo:
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")

    with gr.Row():
        doc_input = gr.File(
            label="Upload project documentation",
            file_types=[".pdf", ".md", ".txt"],
            type="filepath"
        )

        with gr.Column():
            status_md = gr.Markdown(
                f"**Current document:** {knowledge_base['name'] or 'None'}"
            )
            sample_btn = gr.Button("Use the sample document")

    question_box = gr.Textbox(
        label="Question",
        placeholder="For example: How do I contribute to this project?"
    )

    ask_btn = gr.Button("Ask", variant="primary")
    answer_md = gr.Markdown()

    with gr.Accordion("Retrieved passages and sources", open=False):
        sources_md = gr.Markdown()

    doc_input.upload(
        on_upload,
        inputs=doc_input,
        outputs=status_md
    )

    def load_sample():
        status = build_knowledge_base(
            SAMPLE_PAGES,
            "Sample CS 101 syllabus",
            is_pdf=True
        )
        return status

    sample_btn.click(
        load_sample,
        outputs=status_md
    )

    ask_btn.click(
        on_ask,
        inputs=question_box,
        outputs=[answer_md, sources_md]
    )

    question_box.submit(
        on_ask,
        inputs=question_box,
        outputs=[answer_md, sources_md]
    )

demo.launch(share=True)


Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9dce8016cd33224bc0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Next steps
1. Re-read your selected problem statement (2A or 2B) in the Theme 2 document.
2. Replace the sample document and example questions with material for your problem statement.
3. Adapt `ANSWER_RULES`, the settings and the application to meet the requirements.
4. Measure the quality of the answers (see "Expected Deliverables" in the problem statement).
5. Attempt items from the **Good to have** section of the problem statement for additional credit.

**Pointers for common extensions:**
- **Markdown and text files (needed for 2B):** read them with `open(path, encoding="utf-8").read()` and treat each file (or each section under a heading) as one page, storing the file name for citations. Allow them in the application with `file_types=[".pdf", ".md", ".txt"]`.
- **Multiple documents:** use `gr.File(file_count="multiple")`, combine the pages of every file, and store the file name in each chunk for citations.
- **Answer without RAG (for comparison):** call `generate(question)` with no context and show the result next to the RAG answer.
- **Relevance threshold:** in `answer()`, if `found[0]["score"]` is below a chosen value, return "The answer could not be found in the document." without calling Gemma.
- **Conversation memory:** replace the question box with `gr.ChatInterface` and include recent messages in the prompt.

## Troubleshooting

| Issue | Resolution |
|---|---|
| The notebook remains in demonstration mode | Add `GOOGLE_API_KEY` in Secrets, enable **Notebook access**, and re-run the Step 2 cell. |
| "API key was rejected" | Create a new key at aistudio.google.com/apikey and update the secret. |
| "Usage limit reached" | Wait about one minute. Free keys allow a limited number of requests per minute. Team members may each create a key. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| "No text was found" | The PDF contains scanned images only. Use a text-based PDF. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |